In [2]:
#!pip install langchain-huggingface 
#!pip install langchain-ollama

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

W1021 20:46:47.724000 7204 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


# 1. Загружаем локальную модель

## 1.1. С использованием transformers

In [4]:
# Вариант 1: создаем пайплайн через HuggingFacePipline
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 200,
        "do_sample": False,
    },
)

Loading weights:   0%|          | 0/338 [00:01<?, ?it/s]

In [ ]:
# Вариант 2: создаем пайплайн через через transformers

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype="auto",
    device_map="auto",
)

hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=False,
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

In [7]:
response = llm.invoke("Объясни, что такое машинное обучение")
print(response)

Объясни, что такое машинное обучение и как оно работает. Машинное обучение - это раздел искусственного интеллекта, который позволяет компьютерам "учиться" на основе данных, чтобы делать предсказания или принимать решения. Это основная идея машинного обучения: создание алгоритмов, которые могут анализировать большие объемы информации и выявлять закономерности в ней.

Основные этапы работы с машинным обучением:

1. **Извлечение данных**: Сначала нужно собрать данные для обучения модели. Это может быть любая информация, которую можно использовать для прогнозирования или принятия решений.

2. **Подготовка данных**: После сборки данных необходимо их подготовить к использованию. Это может включать в себя обработка, нормализацию и преобразование данных.

3. **Выбор модели**: Затем выбирается подходящий метод машинного обучения. Это может быть линейный


- invoke(): Один запрос
- batch(): Несколько запросов
- stream(): Потоковая выдача результата
- ainvoke(): Асинхронный запрос
- abatch(): Асинхронная обработка набора запросов

## 1.2. C использованием Ollama

In [1]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="qwen2.5:1.5b", temperature=0, num_predict=500)

In [4]:
response = llm.invoke("Объясни, что такое машинное обучение")
print(response.content)

Машинное обучение - это раздел искусственного интеллекта, который включает в себя методы обучения компьютера на основе данных. Этот процесс позволяет компьютеру "учиться" и "думать" самостоятельно, без необходимости напрямую указывать ему, как делать это. 

Основные принципы машинного обучения включают:

1. Обучение: Это процесс, при котором компьютер анализирует данные и learns (обучается) на основе этих данных.

2. Проверка: После обучения, компьютер проверяет свои знания на новых данных, чтобы убедиться, что он действительно понял и усвоил информацию.

3. Обновление: Если компьютер не уверен в своих знаниях, он может "обновлять" свои знания на основе новых данных.

Машинное обучение используется в различных областях, включая:

- Написание текста
- Обработка речи
- Классификацию данных
- Построение моделей машинного обучения
- Анализ данных
- Игровые приложения

В целом, машинное обучение позволяет компьютеру "уметь" делать то, что раньше делали люди, но без необходимости напрямую ук

## 1.3. По API

In [2]:
import os
API_KEY = os.environ.get('YA_API_KEY')
FOLDER_ID = os.environ.get('YA_FOLDER_ID')
MODEL = f"gpt://{FOLDER_ID}/aliceai-llm-flash/latest"

In [3]:
from langchain_community.llms import YandexGPT
llm = YandexGPT(folder_id=FOLDER_ID, api_key=API_KEY)

In [4]:
response = llm.invoke("Объясни, что такое машинное обучение")
print(response)

**Машинное обучение (МО)** — это раздел искусственного интеллекта, который позволяет компьютерам и алгоритмам автоматически улучшаться на основе анализа больших объёмов данных без явного программирования. В основе МО лежит идея, что с помощью алгоритмов можно выявить закономерности и зависимости в данных, а затем использовать эти знания для прогнозирования или принятия решений.

Существует три основных парадигмы машинного обучения:

1. **Обучение с учителем (supervised learning)** — алгоритмы обучаются на размеченных данных, то есть таких, для которых известны правильные ответы. Например, можно обучить модель распознавать кошек на фотографиях, предоставив ей множество изображений с пометками, где кошки есть, а где — нет.

2. **Обучение без учителя (unsupervised learning)** — алгоритмы ищут закономерности в неразмеченных данных. Например, кластеризация — это метод, который позволяет разделить данные на группы (кластеры) по схожим признакам.

3. **Обучение с подкреплением (reinforcement 

# 2. Используем шаблон для промпта

Шаблон позволяет отделить инструкции от данных.

Плейсхолдер {term} заменяется входным значением.

Для разных ролей используются сообщения:
- system - инструкции модели;
- human - запрос пользователя;
- ai - ответ модели;
- tool - результат выполнения инструмента.

В современных chat-моделях результатом промпта часто является не строка, а набор сообщений с ролями.

In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Ты преподаватель машинного обучения. "
        "Объясняй простыми словами и приводи пример."
    ),
    (
        "human",
        "Объясни термин {term}"
    )
])

messages = prompt.invoke({"term": "градиентный бустинг"})
print(messages)

messages=[SystemMessage(content='Ты преподаватель машинного обучения. Объясняй простыми словами и приводи пример.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Объясни термин градиентный бустинг', additional_kwargs={}, response_metadata={})]


# 3. LCEL

LangChain Expression Language

LCEL позволяет соединять совместимые компоненты в цепочки с помощью оператора |

Последовательность вычислений:
1. prompt создаёт сообщения.
2. llm генерирует ответ.
3. StrOutputParser() извлекает текст.
4. invoke() запускает всю последовательность.

В терминах функционального программирования это композиция функций.

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import YandexGPT
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Ты преподаватель машинного обучения. "
        "Объясняй простыми словами и приводи пример."
    ),
    (
        "human",
        "Объясни термин {term}"
    )
])

llm = YandexGPT(folder_id=FOLDER_ID, api_key=API_KEY)

chain = prompt | llm | StrOutputParser()
answer = chain.invoke({"term": "градиентный бустинг"})
print(answer)

Градиентный бустинг — это метод машинного обучения, который помогает улучшить точность предсказаний за счёт комбинации нескольких слабых моделей (например, деревьев решений).

Представьте, что вы хотите предсказать, сколько времени займёт дорога до работы. Вы можете использовать несколько простых моделей, которые учитывают разные факторы, например, расстояние, пробки и время суток. Затем вы объединяете эти модели, чтобы получить более точный прогноз.

В градиентном бустинге слабые модели последовательно обучаются исправлять ошибки предыдущих моделей, что позволяет создать сильную модель, способную делать точные предсказания.


# 4. Парсинг ответа

OutputParser преобразует ответ.

К примеру, модель должна возвращать категорию обращения, а не произвольный текст.

Вместо произвольного ответа она возвращает объект со структурированными полями.

with_structured_output() использует возможности выбранной модели, не любая модель гарантированно вернет корректные данные

In [8]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama

class Ticket(BaseModel):
    category: Literal[
        "billing",
        "technical",
        "other"
    ] = Field(description="Категория обращения")

    priority: Literal[
        "low",
        "medium",
        "high"
    ]


llm = ChatOllama(model="qwen2.5:1.5b", temperature=0, num_predict=500)
structured_llm = llm.with_structured_output(Ticket)
result = structured_llm.invoke("Сервис не запускается после обновления")

print(result.category)
print(result.priority)

technical
high


# 5. RAG в LangChain

Для работы с документами LangChain использует объект Document, который хранит текст и метаданные

In [10]:
from langchain_core.documents import Document

doc = Document(
    page_content=("RAG объединяет поиск по внешним источникам и генерацию ответа LLM."),
    metadata={
        "source": "rag_intro.txt",
        "topic": "RAG"
    }
)

print(doc.page_content)
print(doc.metadata)

RAG объединяет поиск по внешним источникам и генерацию ответа LLM.
{'source': 'rag_intro.txt', 'topic': 'RAG'}


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents([doc])

for chunk in chunks:
    print(chunk.page_content)

W1021 20:02:37.577000 8004 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


RAG объединяет поиск по внешним источникам и генерацию ответа LLM.


In [12]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama

# 1. Исходные документы
docs = [
    Document(
        page_content=(
            "RAG — подход, при котором перед генерацией ответа выполняется поиск по внешним источникам. "
            "Найденные фрагменты передаются языковой модели."
        ),
        metadata={"source": "rag.txt"}
    ),
    Document(
        page_content=(
            "Fine-tuning изменяет параметры модели на примерах обучения."
            "Его применяют для адаптации поведения модели к конкретной задаче."
        ),
        metadata={"source": "finetuning.txt"}
    ),
    Document(
        page_content=(
            "Embeddings — числовые векторные представления текста. "
            "Они позволяют искать документы по смысловому сходству."
        ),
        metadata={"source": "embeddings.txt"}
    )
]

# 2. Разбиение документов
splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=30
)
chunks = splitter.split_documents(docs)

# 3. Локальная embedding-модель
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 4. Создание векторного хранилища
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="ml_knowledge_demo"
)

# 5. Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# 6. Промпт
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Отвечай только на основании контекста. "
        "Если данных недостаточно, скажи об этом.\n\n"
        "Контекст:\n{context}"
    ),
    ("human", "{question}")
])

# # 7. Модель
from langchain_community.llms import YandexGPT
llm = YandexGPT(folder_id=FOLDER_ID, api_key=API_KEY)

# 8. Сборка RAG-цепочки
def format_docs(documents):
    return "\n\n".join(
        f"Источник: {doc.metadata.get('source', 'unknown')}\n"
        f"{doc.page_content}"
        for doc in documents
    )

def retrieve_and_format(inputs):
    documents = retriever.invoke(inputs["question"])
    return {
        "question": inputs["question"],
        "context": format_docs(documents)
    }

rag_chain = (
    retrieve_and_format
    | prompt
    | llm
    | StrOutputParser()
)

# 9. Запрос
question = "Чем RAG отличается от fine-tuning?"
answer = rag_chain.invoke({"question": question})

print(answer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RAG (Retrieval-Augmented Generation) — это подход, при котором перед генерацией ответа выполняется поиск по внешним источникам, и найденные фрагменты передаются языковой модели. Fine-tuning изменяет параметры модели на примерах обучения для адаптации её поведения к конкретной задаче. 

Таким образом, основное отличие между RAG и fine-tuning заключается в том, что RAG использует внешние источники для обогащения генерации ответов, в то время как fine-tuning изменяет внутренние параметры модели для адаптации к конкретной задаче.


# 6. LangChain Tools и агенты

## 6.1. Создаем свой инструмент

In [13]:
from langchain.tools import tool

@tool
def calculate_mean(values: list[float]) -> float:
    """Вычисляет среднее арифметическое списка чисел."""
    if not values:
        raise ValueError("Список не должен быть пустым")
    return sum(values) / len(values)

C:\Users\D\anaconda3\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [14]:
print(calculate_mean.name)
print(calculate_mean.description)
print(calculate_mean.invoke({"values": [10, 20, 30]}))

calculate_mean
Вычисляет среднее арифметическое списка чисел.
20.0


## 6.2. Создаем агента

In [16]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama

@tool
def calculate_mean(values: list[float]) -> float:
    """Вычисляет среднее арифметическое чисел."""
    if not values:
        raise ValueError("Пустой список")
    return sum(values) / len(values)

llm = ChatOllama(model="qwen2.5:1.5b", temperature=0, num_predict=500)

agent = create_agent(
    model=llm,
    tools=[calculate_mean],
    system_prompt=(
        "Ты помощник по анализу данных. "
        "Используй calculate_mean, если требуется вычислить среднее значение."
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Посчитай среднее для 12, 18, 30 и объясни результат."
    }]
})

for message in result["messages"]:
    if message.type == "ai" and message.content:
        print(message.content)

Среднее значение для чисел 12, 18 и 30 равно 1200.0. Это означает, что если бы мы добавили эти числа, их количество, и разделили результат на 3, мы получили бы 1200.0.


In [13]:
response = llm.invoke("Посчитай среднее для 12, 18, 30 и объясни результат.")
print(response.content)

Давайте рассчитаем среднее арифметическое для чисел 12, 18, 30.

Среднее арифметическое - это сумма всех чисел разделенная на количество чисел.

1. Сначала найдем сумму этих чисел:
   12 + 18 + 30 = 60

2. Затем разделим сумму на количество чисел:
   60 ÷ 3 = 20

Таким образом, среднее арифметическое для чисел 12, 18, 30 равно 20.

Объяснение:
Среднее арифметическое показывает, сколько получится, если взять среднее значение всех чисел. В данном случае, если взять среднее значение всех этих чисел, то получится 20. Это значит, что если взять 12, 18, и 30, а затем разделить их сумму на 3, то результат будет 20.


# 7. LangGraph

Основные абстракции LangGraph

LangGraph предназначен для описания приложения как ориентированного графа вычислений. Его базовые понятия:
1) State - состояние: Данные, которые нужны процессу: запрос, найденные документы, история сообщений, результат проверки, число попыток.
2) Node - узел: Python-функция, которая читает состояние, выполняет работу и возвращает обновления
3) Edge - ребро: Определяет, какой узел запустить следующим
4) Conditional Edge - условный переход: Выбирает следующий шаг в зависимости от текущего состояния
5) Checkpointer - сохранение состояния: Сохраняет контрольные точки выполнения, чтобы продолжать работу после прерывания или восстанавливать ход процесса.

## 7.1. Пример графа без LLM

- StateGraph(State): задает структуру состояния.
- add_node(): регистрирует функции.
- add_edge(): задает переходы.
- add_conditional_edges(): добавляет ветвление.
- START и END: определяют начало и завершение графа.
- compile(): подготавливает граф к выполнению.
- invoke(): запускает граф и возвращает финальное состояние.

In [18]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    category: str
    answer: str

def classify(state: State):
    question = state["question"].lower()
    if any(word in question for word in ["оплата", "счет", "деньги"]):
        category = "billing"
    elif any(word in question for word in ["ошибка", "не работает", "сбой"]):
        category = "technical"
    else:
        category = "other"
    return {"category": category}

def billing(state: State):
    return {"answer": "Обращение направлено в отдел оплаты."}

def technical(state: State):
    return {"answer": "Обращение направлено в техподдержку."}

def other(state: State):
    return {"answer": "Обращение направлено в общую поддержку."}

def route(state: State) -> str:
    return state["category"]

builder = StateGraph(State)

builder.add_node("classify", classify)
builder.add_node("billing", billing)
builder.add_node("technical", technical)
builder.add_node("other", other)

builder.add_edge(START, "classify")

builder.add_conditional_edges(
    "classify",
    route,
    {
        "billing": "billing",
        "technical": "technical",
        "other": "other",
    }
)

builder.add_edge("billing", END)
builder.add_edge("technical", END)
builder.add_edge("other", END)

graph = builder.compile()

result = graph.invoke({
    "question": "Оплата счета не прошла",
    "category": "",
    "answer": "",
})

print(result["category"])
print(result["answer"])

billing
Обращение направлено в отдел оплаты.


## 7.2. Добавляем LLM

In [17]:
from typing import Literal
from typing_extensions import TypedDict

from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END

# 1. Состояние графа
class SupportState(TypedDict):
    message: str
    category: str
    response: str

# 2. Схема структурированного ответа классификатора
class Classification(BaseModel):
    category: Literal["billing", "technical", "other"] = Field(
        description="Категория обращения пользователя"
    )
    
# 3. Инициализация LLM
llm = ChatOllama(model="qwen2.5:1.5b", temperature=0, num_predict=500)
classifier = llm.with_structured_output(Classification)

# 4. Узел классификации
def classify(state: SupportState) -> dict:
    result = classifier.invoke([
        (
            "system",
            """
            Ты классификатор обращений в службу поддержки.

            Выбери ровно одну категорию:
            - billing: оплата, счета, возвраты, подписки, списания;
            - technical: ошибки, сбои, проблемы с API, настройкой или работой приложения;
            - other: остальные вопросы.

            Классифицируй по смыслу обращения.
            """
        ),
        ("human", state["message"]),
    ])

    return {"category": result.category}

# 5. Обработчики категорий
def handle_billing(state: SupportState) -> dict:
    return {
        "response": (
            "Ваше обращение отнесено к вопросам оплаты. "
            f"Текст обращения: {state['message']}"
        )
    }

def handle_technical(state: SupportState) -> dict:
    return {
        "response": (
            "Ваше обращение отнесено к техническим вопросам. "
            f"Текст обращения: {state['message']}"
        )
    }

def handle_other(state: SupportState) -> dict:
    return {
        "response": (
            "Ваше обращение отнесено к категории прочих вопросов. "
            f"Текст обращения: {state['message']}"
        )
    }

# 6. Функция маршрутизации
def route_category(
    state: SupportState,
) -> Literal["billing", "technical", "other"]:
    if state["category"] == "billing":
        return "billing"
    elif state["category"] == "technical":
        return "technical"
    else:
        return "other"

# 7. Сборка графа
builder = StateGraph(SupportState)

builder.add_node("classify", classify)
builder.add_node("billing", handle_billing)
builder.add_node("technical", handle_technical)
builder.add_node("other", handle_other)

builder.add_edge(START, "classify")

builder.add_conditional_edges(
    "classify",
    route_category,
    {
        "billing": "billing",
        "technical": "technical",
        "other": "other",
    },
)

builder.add_edge("billing", END)
builder.add_edge("technical", END)
builder.add_edge("other", END)

graph = builder.compile()

# 8. Проверка
messages = [
    "С меня дважды списали деньги за подписку",
    "При запуске API получаю ошибку 500",
    "Как связаться с вашей командой?",
]

for message in messages:
    result = graph.invoke({
        "message": message,
        "category": "",
        "response": "",
    })

    print("Вопрос:", message)
    print("Категория:", result["category"])
    print("Ответ:", result["response"])
    print("-" * 60)

Вопрос: С меня дважды списали деньги за подписку
Категория: billing
Ответ: Ваше обращение отнесено к вопросам оплаты. Текст обращения: С меня дважды списали деньги за подписку
------------------------------------------------------------
Вопрос: При запуске API получаю ошибку 500
Категория: technical
Ответ: Ваше обращение отнесено к техническим вопросам. Текст обращения: При запуске API получаю ошибку 500
------------------------------------------------------------
Вопрос: Как связаться с вашей командой?
Категория: other
Ответ: Ваше обращение отнесено к категории прочих вопросов. Текст обращения: Как связаться с вашей командой?
------------------------------------------------------------


## 7.3. Маршрутизация запросов

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableBranch

llm = ChatOllama(model="qwen2.5:1.5b", temperature=0, num_predict=500)

classifier_prompt = ChatPromptTemplate.from_template("""
Определи категорию запроса.

Доступные категории:
rag — вопросы о документах и базе знаний;
code — написание и объяснение программного кода;
general — остальные вопросы.

Верни только название категории.

Запрос: {question}
""")

classifier = classifier_prompt | llm | StrOutputParser()

rag_chain = (
    ChatPromptTemplate.from_template(
        "Ответь на вопрос о базе знаний: {question}"
    )
    | llm
    | StrOutputParser()
)

code_chain = (
    ChatPromptTemplate.from_template(
        "Ты Python-разработчик. Ответь на вопрос: {question}"
    )
    | llm
    | StrOutputParser()
)

general_chain = (
    ChatPromptTemplate.from_template(
        "Ответь на вопрос пользователя: {question}"
    )
    | llm
    | StrOutputParser()
)

def normalize_category(value: str) -> str:
    value = value.strip().lower()
    for category in ("rag", "code", "general"):
        if category in value:
            return category
    return "general"

router = RunnableBranch(
    (
        lambda x: normalize_category(x["category"]) == "rag",
        rag_chain,
    ),
    (
        lambda x: normalize_category(x["category"]) == "code",
        code_chain,
    ),
    general_chain,
)

application = (
    RunnableLambda(
        lambda x: {
            **x,
            "category": normalize_category(
                classifier.invoke({"question": x["question"]})
            ),
        }
    )
    | router
)

result = application.invoke({
    "question": "Напиши функцию Python для чтения CSV-файла"
})

print(result)

Конечно! Вот пример функции, которая читает CSV-файл и возвращает его содержимое в виде списка списков:

```python
import csv

def read_csv_file(file_path):
    with open(file_path, mode='r', encoding='utf-8') as file:
        csv_reader = csv.reader(file)
        data = [row for row in csv_reader]
        return data

# Пример использования
file_path = 'path/to/your/file.csv'
data = read_csv_file(file_path)
print(data)
```

Эта функция открывает файл на чтение, создает объект `csv.reader`, который читает файл построчно, и затем возвращает список списков, где каждый элемент списка - это строка из файла.

Если у вас есть конкретный файл, который нужно прочитать, просто замените `file_path` на путь к вашему файлу.


## 7.4. Планировщик запросов

In [19]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import YandexGPT

llm = YandexGPT(folder_id=FOLDER_ID, api_key=API_KEY)

class PlanState(TypedDict):
    task: str
    steps: list[str]
    results: list[str]
    final_answer: str

planner = (
    ChatPromptTemplate.from_template("""
Составь план решения задачи: {task}

Верни 3 коротких шага, каждый с новой строки.
Не используй нумерацию.
""")
    | llm
    | StrOutputParser()
)

executor = (
    ChatPromptTemplate.from_template("""
        Задача: {task}
        Текущий шаг: {step}
        
        Выполни только этот шаг. Дай конкретный результат.
""")
    | llm
    | StrOutputParser()
)

def make_plan(state: PlanState):
    text = planner.invoke({"task": state["task"]})
    steps = [
        line.strip(" -\t")
        for line in text.splitlines()
        if line.strip()
    ]
    return {
        "steps": steps[:5],
        "results": [],
    }

def execute_steps(state: PlanState):
    results = []
    for step in state["steps"]:
        result = executor.invoke({
            "task": state["task"],
            "step": step,
        })
        results.append(result)
    return {"results": results}

def synthesize(state: PlanState):
    prompt = ChatPromptTemplate.from_template("""
Исходная задача: {task}

Результаты выполнения шагов:
{results}

Составь единый связный ответ. Устрани повторы.
""")
    chain = prompt | llm | StrOutputParser()

    return {
        "final_answer": chain.invoke({
            "task": state["task"],
            "results": "\n\n".join(state["results"]),
        })
    }

builder = StateGraph(PlanState)

builder.add_node("planner", make_plan)
builder.add_node("executor", execute_steps)
builder.add_node("synthesizer", synthesize)

builder.add_edge(START, "planner")
builder.add_edge("planner", "executor")
builder.add_edge("executor", "synthesizer")
builder.add_edge("synthesizer", END)

graph = builder.compile()

result = graph.invoke({
    "task": "Подготовить обзор применения RAG в корпоративных системах",
    "steps": [],
    "results": [],
    "final_answer": "",
})

print("План:")
for step in result["steps"]:
    print("-", step)

print("\nИтог:")
print(result["final_answer"])

План:
- Определить основные области применения RAG (Retrieval-Augmented Generation) в корпоративных системах.
- Изучить примеры использования RAG для решения бизнес-задач в различных отраслях.
- Сформулировать выводы о потенциале и преимуществах применения RAG в корпоративных системах.

Итог:
**Применение RAG (Retrieval-Augmented Generation) в корпоративных системах**

RAG (Retrieval-Augmented Generation) — это технология, которая улучшает качество генерации текста за счёт интеграции внешних данных. Она находит применение в различных корпоративных системах и процессах.

**Основные области применения RAG:**

1. **Обработка естественного языка (NLP) и чат-боты.** RAG улучшает качество ответов в корпоративных чат-ботах, извлекая релевантную информацию из внутренних баз данных и документов.
2. **Поддержка клиентов.** Системы поддержки клиентов могут предоставлять более точные и персонализированные ответы на запросы клиентов через извлечение соответствующих данных из корпоративных баз знани

## 7.5. Генерация с проверкой качества

In [20]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import YandexGPT

llm = YandexGPT(folder_id=FOLDER_ID, api_key=API_KEY)

class ReviewState(TypedDict):
    task: str
    draft: str
    feedback: str
    attempts: int
    approved: bool

generator = (
    ChatPromptTemplate.from_template("""
Выполни задачу:
{task}

Замечания к предыдущему варианту:
{feedback}

Если замечаний нет, подготовь первый вариант.
""")
    | llm
    | StrOutputParser()
)

evaluator = (
    ChatPromptTemplate.from_template("""
Проверь ответ по задаче.

Задача: {task}
Ответ: {draft}

Если ответ удовлетворяет задаче, первая строка должна быть
ровно APPROVED.

Иначе первая строка должна быть ровно REVISE,
а далее перечисли конкретные недостатки.
""")
    | llm
    | StrOutputParser()
)

def generate(state: ReviewState):
    draft = generator.invoke({
        "task": state["task"],
        "feedback": state["feedback"] or "Нет.",
    })
    return {
        "draft": draft,
        "attempts": state["attempts"] + 1,
    }

def evaluate(state: ReviewState):
    feedback = evaluator.invoke({
        "task": state["task"],
        "draft": state["draft"],
    })

    approved = feedback.strip().startswith("APPROVED")
    return {
        "approved": approved,
        "feedback": feedback,
    }

def route_after_review(state: ReviewState):
    if state["approved"] or state["attempts"] >= 3:
        return END
    return "generate"

builder = StateGraph(ReviewState)

builder.add_node("generate", generate)
builder.add_node("evaluate", evaluate)

builder.add_edge(START, "generate")
builder.add_edge("generate", "evaluate")
builder.add_conditional_edges(
    "evaluate",
    route_after_review,
    {
        "generate": "generate",
        END: END,
    },
)

review_graph = builder.compile()

result = review_graph.invoke({
    "task": (
        "Объясни разницу между fine-tuning и RAG. "
        "Приведи два практических примера."
    ),
    "draft": "",
    "feedback": "",
    "attempts": 0,
    "approved": False,
})

print("Число попыток:", result["attempts"])
print("Проверка пройдена:", result["approved"])
print("\nОтвет:\n", result["draft"])

Число попыток: 1
Проверка пройдена: True

Ответ:
 **Fine-tuning** (дообучение) и **RAG** (Retrieval-Augmented Generation) — это два разных подхода в области искусственного интеллекта и обработки естественного языка.

**Fine-tuning** — это процесс настройки параметров предварительно обученной модели на новом наборе данных. Цель fine-tuning — адаптировать модель к конкретной задаче или доработать её для работы с определённым типом данных. Например, модель для обработки текстов на английском языке можно дообучить на текстах на другом языке или в специфическом стиле (например, научном или деловом).

**RAG** — это метод, который использует предварительно обученную модель для генерации ответов, но при этом учитывает контекст и информацию из внешних источников. RAG включает в себя два основных компонента: систему поиска (retrieval) и генеративную модель (например, языковая модель). Система поиска находит релевантные документы или фрагменты текста, а генеративная модель использует эту информац

## 7.6. Перепроверка релевантности источника в RAG

### a) Собираем vectorstore

In [ ]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Исходные документы
docs = [
    Document(
        page_content=(
            "RAG — подход, при котором перед генерацией ответа выполняется поиск по внешним источникам. "
            "Найденные фрагменты передаются языковой модели."
        ),
        metadata={"source": "rag.txt"}
    ),
    Document(
        page_content=(
            "Fine-tuning изменяет параметры модели на примерах обучения."
            "Его применяют для адаптации поведения модели к конкретной задаче."
        ),
        metadata={"source": "finetuning.txt"}
    ),
    Document(
        page_content=(
            "Embeddings — числовые векторные представления текста. "
            "Они позволяют искать документы по смысловому сходству."
        ),
        metadata={"source": "embeddings.txt"}
    )
]

# 2. Разбиение документов
splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=30
)
chunks = splitter.split_documents(docs)

# 3. Локальная embedding-модель
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 4. Создание векторного хранилища
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="ml_knowledge_demo"
)

### б) Создаем класс состояния графа

In [1]:
from typing import TypedDict
from langchain_core.documents import Document

class RAGState(TypedDict):
    question: str
    search_query: str
    documents: list[Document]
    relevant_documents: list[Document]
    scores: list[float]
    answer: str
    iteration: int
    sufficient_context: bool

### в) Грузим модель reranker 

In [3]:
import os
from sentence_transformers import CrossEncoder
from huggingface_hub import login

HF_DOWNLOAD_TOKEN = os.environ.get('HF_DOWNLOAD_TOKEN')

# Cross-encoder reranker
login(token=HF_DOWNLOAD_TOKEN) # Можно зарегистрироваться в HF и получить токен, тогда качает быстрее и без ограничений, но можно и без него
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

MAX_ITERATIONS = 3
TOP_K = 5
MIN_RELEVANT_DOCS = 2

# Примерный порог, подбирается для каждой выборки индивидуально
RELEVANCE_THRESHOLD = 0.3

W1022 00:12:05.698000 5140 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\D\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\D\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(messa

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

C:\Users\D\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\D\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

### г) Создаем узлы графа (функции)

In [19]:
def retrieve(state: RAGState) -> dict:
    k = TOP_K + state["iteration"] * 3

    docs = vectorstore.similarity_search(
        state["search_query"],
        k=k
    )

    return {"documents": docs}

def rerank(state: RAGState) -> dict:
    docs = state["documents"]
    question = state["question"]

    if not docs:
        return {
            "relevant_documents": [],
            "scores": [],
            "sufficient_context": False
        }

    pairs = [
        (question, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(docs, scores),
        key=lambda item: float(item[1]),
        reverse=True
    )

    # Оставляем только документы выше порога
    relevant = [
        doc
        for doc, score in ranked
        if float(score) >= RELEVANCE_THRESHOLD
    ]

    # Проверяем, достаточно ли контекста
    sufficient = len(relevant) >= MIN_RELEVANT_DOCS

    return {
        "relevant_documents": relevant,
        "scores": [float(score) for _, score in ranked],
        "sufficient_context": sufficient
    }

def retry_search(state: RAGState) -> dict:
    prompt = f"""
    Ты помогаешь улучшить поисковый запрос для RAG-системы.
    
    Исходный вопрос:
    {state["question"]}
    
    Предыдущий запрос:
    {state["search_query"]}
    
    Предыдущий поиск не нашел достаточно релевантного контекста.
    
    Сформулируй альтернативный поисковый запрос:
    - сохрани исходный смысл вопроса;
    - добавь важные термины и синонимы;
    - не отвечай на сам вопрос;
    - верни только поисковый запрос.
    """

    response = llm.invoke(prompt)

    search_query = (
        response.content.strip()
        if hasattr(response, "content")
        else str(response).strip()
    )
    
    return {
        "search_query": search_query,
        "iteration": state["iteration"] + 1
    }

def generate_answer(state: RAGState) -> dict:
    context = "\n\n".join(
        doc.page_content
        for doc in state["relevant_documents"]
    )

    prompt = f"""
    Ответь на вопрос, используя только предоставленный контекст.
    
    Если контекст не содержит ответа, прямо сообщи об этом.
    Не придумывай факты. По возможности укажи источники из метаданных документов.
    
    Вопрос:
    {state["question"]}
    
    Контекст:
    {context}
    """

    response = llm.invoke(prompt)

    return {"answer": response.content}

def decide_next_step(state: RAGState) -> str:
    if state["sufficient_context"]:
        return "generate"

    if state["iteration"] + 1 >= MAX_ITERATIONS:
        return "fallback"

    return "retry"

def fallback(state: RAGState) -> dict:
    return {
        "answer": (
            "Не удалось найти достаточно релевантного контекста после нескольких попыток поиска. "
            "Не могу надежно ответить на вопрос по найденным данным."
        )
    }

### д) Собираем граф

In [17]:
from langgraph.graph import StateGraph, START, END
from langchain_community.llms import YandexGPT

# LLM для переформулирования запроса и ответа
API_KEY = os.environ.get('YA_API_KEY')
FOLDER_ID = os.environ.get('YA_FOLDER_ID')
MODEL = f"gpt://{FOLDER_ID}/aliceai-llm-flash/latest"

llm = YandexGPT(folder_id=FOLDER_ID, api_key=API_KEY)

builder = StateGraph(RAGState)

builder.add_node("retrieve", retrieve)
builder.add_node("rerank", rerank)
builder.add_node("retry", retry_search)
builder.add_node("generate", generate_answer)
builder.add_node("fallback", fallback)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "rerank")

builder.add_conditional_edges(
    "rerank",
    decide_next_step,
    {
        "generate": "generate",
        "retry": "retry",
        "fallback": "fallback"
    }
)

builder.add_edge("retry", "retrieve")
builder.add_edge("generate", END)
builder.add_edge("fallback", END)

rag_graph = builder.compile()

### е) Пробуем в работе

In [18]:
result = rag_graph.invoke({
    "question": "Как работает механизм долгосрочной памяти LangGraph?",
    "search_query": "LangGraph long-term memory Store namespace",
    "documents": [],
    "relevant_documents": [],
    "scores": [],
    "answer": "",
    "iteration": 0,
    "sufficient_context": False
})

print(result["answer"])

Не удалось найти достаточно релевантного контекста после нескольких попыток поиска. Не могу надежно ответить на вопрос по найденным данным.
